In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker
from pathlib import Path
import json

# NEW DATA

## CODE

In [ ]:
def extract_L_vs_key_time(data_dict, key):
    L_values = []
    time_values = []
    for L, entry in data_dict.items():
        if key in entry:
            L_values.append(L)
            time_values.append((entry[key][0] + entry[key][1]) / 2)  # Averaging over the two values
    return L_values, time_values

In [ ]:
def plot_loglog_fit(data_dict, key, key_vs_label, base_name, ax=None):
    x, y = extract_L_vs_key_time(data_dict, key)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    y_label = key_vs_label[key]

    # Note that the timing scales by atleast L^4 so we have some weights to give more importance to the larger L values in the fit
    weights = np.linspace(min(x)**2, max(x)**2, len(x))
    slope, intercept = np.polyfit(np.log(x), np.log(y), 1, w=weights)
    fit_y = np.exp(intercept) * x**slope

    if ax is None:
        _, ax = plt.subplots()

    ax.plot(x, fit_y, label=rf'fit: $\propto L^{{{slope:.2f}}}$',linestyle=':',color='green')
    ax.scatter(x, y, label=f'{y_label}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.xaxis.set_major_locator(ticker.FixedLocator(x))
    ax.xaxis.set_major_formatter(ticker.FixedFormatter([str(int(val)) for val in x]))
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    ax.set_xlabel('L')
    ax.set_ylabel(y_label)
    ax.set_title(f'L vs {y_label} on {base_name}')
    ax.text(0.05, 0.95, rf'Slope $\approx {slope:.2f}$', transform=ax.transAxes, va='top')
    ax.legend()

    return slope, intercept, ax


def plot_loglog_fit_all_in_one(data_dict, key_vs_label, base_name, ax=None):

    for key in key_vs_label.keys():
        x,y = extract_L_vs_key_time(data_dict, key)
        y_label = key_vs_label[key]

        # Note that the timing scales by atleast L^4 so we have some weights to give more importance to the larger L values in the fit
        weights = np.linspace(min(x)**2, max(x)**2, len(x))
        slope, intercept = np.polyfit(np.log(x), np.log(y), 1, w=weights)
        fit_y = np.exp(intercept) * x**slope

        if ax is None:
            _, ax = plt.subplots()

        ax.plot(x, fit_y)
        ax.scatter(x, y, label=rf'{y_label}: $\propto L^{{{slope:.2f}}}$',linestyle=':')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.xaxis.set_major_locator(ticker.FixedLocator(x))
    ax.xaxis.set_major_formatter(ticker.FixedFormatter([str(int(val)) for val in x]))
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    ax.set_xlabel('L')
    ax.set_ylabel('Time (s)')
    ax.set_title(f'Scaling with L on {base_name}')
    # ax.text(0.05, 0.95, rf'Slope $\approx {slope:.2f}$', transform=ax.transAxes, va='top')
    ax.legend()

## PLOTS

In [ ]:
# IMPORTANT note that arpack_dnaupd_wrapper_time contains LU_time as well so subtract it to get the pure ARPACK time

base_name , json_timing_data = "PC", Path('pc_timing_data.json')
# base_name , json_timing_data = "HPC", Path('hpc_timing_data.json')


if json_timing_data.exists():
    data_dict = json.load(json_timing_data.open())
    # Keys are strings when read from JSON, convert them back to integers
    data_dict = {int(k): v for k, v in data_dict.items()}
    # get just the timing data for ARPACK by subtracting LU_time from arpack_dnaupd_wrapper_time
    for L, entry in data_dict.items():
        if 'arpack_dnaupd_wrapper_time' in entry and 'LU_time' in entry:
            arpack_time = entry['arpack_dnaupd_wrapper_time']
            lu_time = entry['LU_time']
            pure_arpack_time = [arpack - lu for arpack, lu in zip(arpack_time, lu_time)]
            data_dict[L]['arpack_dnaupd_wrapper_time'] = pure_arpack_time

key_vs_label = {}
key_vs_label['generate_matrices_time']= "Matrix Generation Time (s)"
key_vs_label['dggev_time']= "DGGEV Time (s)"
key_vs_label['LU_time']= "LU Decomposition Time (s)"
key_vs_label['arpack_dnaupd_wrapper_time']= "ARPACK(DNAUPD) Time (s)"



In [ ]:
save_path = Path('images/temp').resolve()
if not save_path.exists():
    save_path.mkdir(parents=True)
save_path

In [ ]:
y_key = "LU_time"

plot_loglog_fit(data_dict, y_key, key_vs_label, base_name)

In [ ]:
with plt.style.context('ggplot'):
    for y_col in key_vs_label.keys():
        plot_loglog_fit(data_dict, y_col, key_vs_label, base_name)
        plt.savefig(save_path / f'{base_name}_{y_col.lower()}.png', dpi=300)  # Save the figure
        # plt.show()
    plot_loglog_fit_all_in_one(data_dict, key_vs_label, base_name)
    plt.savefig(save_path / f'{base_name}_all_in_one.png', dpi=300)  # Save the figure

# OLD DATA

In [ ]:
# OLD DATA
# L , MatGen , Arpack, LU, dggev
wheeler = [[15 , 0.071161 , 0.000712  , 3.26000e-04, 4.52780000e-02],
[25 , 0.640234 , 0.004513  , 4.48400e-03, 1.46575600e+00],
[35 , 2.72389 , 0.015021  , 3.04950e-02, 2.27840010e+01],
[45 , 9.188617 , 0.094585  , 1.22304e-01, 1.50830308e+02],
[55 , 21.931774 , 0.230849  , 4.14148e-01, 6.44315238e+02]]
wheeler = np.array(wheeler)



pc = [[15, 0.087328 , 0.003085  , 3.623000e-03  , 0.104892],
[25, 0.63935 , 0.017232  , 4.547300e-02  , 1.806011],
[35, 2.83385 , 0.05992   , 3.769910e-01  , 15.621133],
[45, 10.355544 , 0.196789  , 2.010160      , 74.043790],
[55, 21.472547 , 0.481778  , 6.144312      , 275.292717]]
pc = np.array(pc)

labels = ['L', 'MatGen', 'Arpack', 'LU', 'dggev']


In [ ]:
save_path = Path('images/temp').resolve()
save_path

In [ ]:
with plt.style.context('ggplot'):
    plt.scatter(wheeler[:, 0], wheeler[:, 1], label='MatGen')
    plt.yscale('log')
    plt.xscale('log')

In [ ]:
def plot_loglog_fit(data, y_col, ax=None):
    x = data[:, labels.index('L')].astype(float)

    if isinstance(y_col, str):
        y_idx = labels.index(y_col)
        y_label = y_col
    else:
        y_idx = y_col
        y_label = labels[y_idx]

    y = data[:, y_idx].astype(float)
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    weights = np.linspace(1.0, 2.0, len(x))
    slope, intercept = np.polyfit(np.log(x), np.log(y), 1, w=weights)
    fit_y = np.exp(intercept) * x**slope

    if ax is None:
        _, ax = plt.subplots()

    ax.plot(x, fit_y, label=rf'fit: $\propto L^{{{slope:.2f}}}$',linestyle=':',color='green')
    ax.scatter(x, y, label=f'{y_label}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.xaxis.set_major_locator(ticker.FixedLocator(x))
    ax.xaxis.set_major_formatter(ticker.FixedFormatter([str(int(val)) for val in x]))
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    ax.xaxis.set_minor_formatter(ticker.NullFormatter())
    ax.set_xlabel('L')
    ax.set_ylabel(y_label)
    ax.set_title(f'L vs {y_label}')
    ax.text(0.05, 0.95, rf'Slope $\approx {slope:.2f}$', transform=ax.transAxes, va='top')
    ax.legend()

    return slope, intercept, ax

In [ ]:
with plt.style.context('ggplot'):
    for y_col in ['MatGen', 'Arpack', 'LU', 'dggev']:
        plot_loglog_fit(wheeler, y_col)
        plt.savefig(save_path / f'wheeler_{y_col.lower()}.png', dpi=300)  # Save the figure
        plt.show()

In [ ]:
with plt.style.context('ggplot'):
    for y_col in ['MatGen', 'Arpack', 'LU', 'dggev']:
        plot_loglog_fit(pc, y_col)
        plt.savefig(save_path / f'pc_{y_col.lower()}.png', dpi=300)  # Save the figure
        plt.show()

In [ ]:
with plt.style.context('ggplot'):

    for data,label in zip([wheeler, pc], ['Wheeler', 'PC']):

        fig, ax = plt.subplots()

        for i, y_col in enumerate(['MatGen', 'Arpack', 'LU', 'dggev']):
            y_idx = labels.index(y_col)
            x = data[:, labels.index('L')]
            y = data[:, y_idx]

            slope, intercept = np.polyfit(np.log(x), np.log(y), 1)
            fit_y = np.exp(intercept) * x**slope

            ax.scatter(x, y, label=y_col)
            ax.plot(x, fit_y, linestyle=':')
            ax.text(
                0.05,
                0.95 - 0.08 * i,
                rf'{y_col}: slope $\approx {slope:.2f}$',
                transform=ax.transAxes,
                va='top',
            )

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.xaxis.set_major_locator(ticker.FixedLocator(x))
        ax.xaxis.set_major_formatter(ticker.FixedFormatter([str(int(val)) for val in x]))
        ax.xaxis.set_minor_locator(ticker.NullLocator())
        ax.xaxis.set_minor_formatter(ticker.NullFormatter())
        ax.set_xlabel('L')
        ax.set_ylabel('Time')
        ax.set_title('L vs Time')
        ax.legend()

        plt.savefig(save_path / f'{label.lower()}_all_loglog.png', dpi=300)
        plt.show()
